# MLOps Assignment 2 — DistilBERT Goodreads Genre Classifier
**IIT Jodhpur | PGD AI Programme | MLOps**

Fine-tunes `distilbert-base-cased` on UCSD Goodreads reviews to classify books into 7 genres.  
Experiment tracking with **Weights & Biases** and model hosting on **Hugging Face Hub**.

---
**Links:**
- GitHub: https://github.com/g25ait2004-cyber/Abhishek
- W&B Dashboard: https://wandb.ai/g25ait2032-iit-jodhpur/mlops-assignment2
- Hugging Face: https://huggingface.co/g25ait2004/DistilBERT_Goodreads

## Step 0 — Install Dependencies

In [ ]:
!pip install -q transformers datasets wandb huggingface_hub scikit-learn

## Step 1 — Load Secrets (Kaggle Secrets — no hardcoded tokens)

Add your `HF_TOKEN` and `WANDB_API_KEY` via **Kaggle → Settings → Secrets** before running.

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hf_token")
secret_value_1 = user_secrets.get_secret("wandb_token")


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# Read tokens from Kaggle Secrets
hf_token    = secrets.get_secret("HF_TOKEN")
wandb_token = secrets.get_secret("WANDB_API_KEY")

os.environ["HF_TOKEN"]      = hf_token
os.environ["WANDB_API_KEY"] = wandb_token

print("Secrets loaded successfully (tokens are hidden).")

## Step 2 — Clone the Project Repository

In [ ]:
import os, shutil

os.chdir('/kaggle/working')

# Remove stale clone if present
if os.path.exists('Abhishek'):
    shutil.rmtree('Abhishek')

!git clone https://github.com/g25ait2004-cyber/Abhishek.git
os.chdir('/kaggle/working/Abhishek')

print("Current directory:", os.getcwd()) 

!ls

## Step 3 — Data Loading, Sampling & Encoding (data.py)

Downloads reviews from the UCSD Goodreads dataset and builds train/test splits.

In [ ]:
# ── Run Refactored Data Pipeline ─────────────────────────────────────────────
MODEL_NAME = "distilbert-base-cased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# 1. Load Data (Fixed the parameter name from reviews_per_genre to sample_size)
genre_reviews = load_all_genres(sample_size=1000) 
tr_texts, tr_labels, te_texts, te_labels = train_test_split_genres(genre_reviews, reviews_per_genre=1000)

# 2. Build explicit mappings out of global space
label2id, id2label = build_label_maps(tr_labels)

# 3. Tokenize and construct optimized PyTorch sets
train_dataset, test_dataset = encode_datasets(
    tr_texts, tr_labels, te_texts, te_labels, tokenizer, label2id
)

print(f"\n🚀 Pipeline complete successfully.")
print(f"├── Train dataset size : {len(train_dataset)}")
print(f"├── Test dataset size  : {len(test_dataset)}")
print(f"└── Target Label map   : {id2label}")

In [ ]:
import gzip, json, pickle, random, requests
from transformers import DistilBertTokenizerFast
import torch
from sklearn.metrics import accuracy_score, f1_score

# ── Label maps ──────────────────────────────────────────────────────────────
label2id: dict = {}
id2label: dict = {}

def build_label_maps(labels):
    global label2id, id2label
    unique = sorted(set(labels))
    label2id = {lbl: idx for idx, lbl in enumerate(unique)}
    id2label  = {idx: lbl for lbl, idx in label2id.items()}
    return label2id, id2label


# ── Dataset class ────────────────────────────────────────────────────────────
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# ── Metrics ──────────────────────────────────────────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }


# ── Data URLs ─────────────────────────────────────────────────────────────────
GENRE_URL_DICT = {
    "poetry":                 "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz",
    "comics_graphic":         "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz",
    "fantasy_paranormal":     "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz",
    "history_biography":      "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz",
    "mystery_thriller_crime": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz",
    "romance":                "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz",
    "young_adult":            "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz",
    "children":                "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz"
}


def load_reviews(url, head=10000, sample_size=2000):
    reviews  = []
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with gzip.open(response.raw, "rt", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if head is not None and i >= head:
                break
            d = json.loads(line)
            reviews.append(d["review_text"])
    return random.sample(reviews, min(sample_size, len(reviews)))


def load_all_genres(pickle_path="genre_reviews_dict.pickle", head=10000, sample_size=2000):
    try:
        print(f"Loading cached data from {pickle_path} ...")
        return pickle.load(open(pickle_path, "rb"))
    except FileNotFoundError:
        pass
    genre_reviews_dict = {}
    for genre, url in GENRE_URL_DICT.items():
        print(f"Downloading reviews for genre: {genre}")
        genre_reviews_dict[genre] = load_reviews(url, head=head, sample_size=sample_size)
    pickle.dump(genre_reviews_dict, open(pickle_path, "wb"))
    print(f"Saved to {pickle_path}")
    return genre_reviews_dict


def train_test_split_genres(genre_reviews_dict, reviews_per_genre=1000, train_frac=0.8):
    train_texts, train_labels = [], []
    test_texts,  test_labels  = [], []
    for genre, reviews in genre_reviews_dict.items():
        sampled = random.sample(reviews, min(reviews_per_genre, len(reviews)))
        split   = int(len(sampled) * train_frac)
        for text in sampled[:split]:
            train_texts.append(text);  train_labels.append(genre)
        for text in sampled[split:]:
            test_texts.append(text);   test_labels.append(genre)
    print(f"Train: {len(train_texts)}  |  Test: {len(test_texts)}  |  Genres: {len(genre_reviews_dict)}")
    return train_texts, train_labels, test_texts, test_labels


def encode_datasets(tr_texts, tr_labels, te_texts, te_labels, tokenizer, max_length=512):
    l2i, _ = build_label_maps(tr_labels)
    tr_enc  = tokenizer(tr_texts, truncation=True, padding=True, max_length=max_length)
    te_enc  = tokenizer(te_texts, truncation=True, padding=True, max_length=max_length)
    return (
        MyDataset(tr_enc, [l2i[y] for y in tr_labels]),
        MyDataset(te_enc, [l2i[y] for y in te_labels]),
    )


# ── Run data pipeline ────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-cased"
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

genre_reviews = load_all_genres()
tr_texts, tr_labels, te_texts, te_labels = train_test_split_genres(genre_reviews)
train_dataset, test_dataset = encode_datasets(tr_texts, tr_labels, te_texts, te_labels, tokenizer)

print(f"\nTrain dataset size : {len(train_dataset)}")
print(f"Test  dataset size : {len(test_dataset)}")
print(f"Number of labels   : {len(id2label)}")
print("Label map:", id2label)

## Step 4 — Load Pre-trained DistilBERT Model

In [ ]:
import wandb
from transformers import (
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

NUM_LABELS   = len(id2label)
DEVICE       = "cuda"  # Kaggle T4 GPU

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
).to(DEVICE)

print(f"Model loaded — {NUM_LABELS} output classes")
print("Classes:", list(id2label.values()))

## Step 5 — Training with W&B Tracking (train.py)

Initialises W&B, configures `TrainingArguments`, and runs fine-tuning.

In [ ]:
import os
import wandb

# Replaced with your W&B project and Hugging Face repo details
WANDB_PROJECT = "MLOps Assignment"
WANDB_RUN     = "distilbert-run-kaggle"
HF_REPO       = "g25ait2004/DistilBERT_Goodreads"

# ── Hyperparameters ──────────────────────────────────────────────────────────
NUM_EPOCHS    = 3
BATCH_TRAIN   = 16
BATCH_EVAL    = 32
LEARNING_RATE = 3e-5
WARMUP_STEPS  = 100
WEIGHT_DECAY  = 0.01
MAX_LENGTH    = 512

# ── W&B init ─────────────────────────────────────────────────────────────────
wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN,
    config={
        "model":         MODEL_NAME,
        "epochs":        NUM_EPOCHS,
        "batch_size":    BATCH_TRAIN,
        "learning_rate": LEARNING_RATE,
        "max_length":    MAX_LENGTH,
        "dataset":       "UCSD Goodreads",
    },
)

# ── Training arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name=WANDB_RUN,
    learning_rate=LEARNING_RATE,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training ...")
trainer.train()
print("Training complete.")

## Step 6 — Save Model Locally & Push to Hugging Face Hub

In [ ]:
import os
from huggingface_hub import login
import wandb

SAVE_DIR = "./DistilBERT_Goodreads"

# Save locally
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved locally to {SAVE_DIR}")

# Push to Hugging Face Hub
login(token=os.environ["HF_TOKEN"])
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

hf_url = f"https://huggingface.co/{HF_REPO}"

# Check if a run is active; if not, use the last run's summary
if wandb.run is not None:
    wandb.run.summary["huggingface_model"] = hf_url
else:
    # 1. Run ID from your WandB dashboard
    RUN_ID = "zvs34qon" 
    
    # 2. Your W&B entity (username/workspace)
    ENTITY = "g25ait2004-indian-institute-of-technology-jodhpur" 
    
    api = wandb.Api()
    # This will construct: g25ait2004-indian-institute-of-technology-jodhpur/MLOps Assignment/zvs34qon
    run = api.run(f"{ENTITY}/{WANDB_PROJECT}/{RUN_ID}") 
    run.summary["huggingface_model"] = hf_url
    run.update()

print(f"Model pushed to: {hf_url}")

## Step 7 — Final Evaluation & W&B Artifact Upload (eval.py)

In [ ]:
import json
from sklearn.metrics import classification_report
import wandb

# 1. Re-initialize the run so the Trainer has somewhere to log
wandb.init(
    project=WANDB_PROJECT,
    id="zvs34qon",        # Updated with your specific run ID
    resume="must"         # This attaches to your previous training run
)

# 2. Now run evaluation
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

# ── Log metrics to W&B ───────────────────────────────────────────────────────
# These manual logs will now work because wandb.run is active again
wandb.log({
    "final/loss":     eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1":       eval_results["eval_f1"],
})

# ── Full classification report ────────────────────────────────────────────────
pred_output = trainer.predict(test_dataset)
preds       = pred_output.predictions.argmax(-1).flatten().tolist()
preds_str   = [id2label[p] for p in preds]

report = classification_report(te_labels, preds_str, output_dict=True)
print("\nClassification Report:")
print(classification_report(te_labels, preds_str))

# ── Save report & upload to W&B as Artifact ───────────────────────────────────
report_path = "eval_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

artifact = wandb.Artifact("eval-report", type="evaluation")
artifact.add_file(report_path)
wandb.log_artifact(artifact)
print(f"Saved and uploaded {report_path} as W&B Artifact.")

wandb.finish()
print("Evaluation complete.")

## Step 8 — Quick Inference Demo

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model=HF_REPO)

test_samples = [
    "The detective found a fingerprint on the cold brass handle.",
    "Their eyes met across the crowded ballroom and she felt her heart race.",
    "In the quiet of the night, the soul speaks in rhyme.",
    "The 19th-century industrial revolution reshaped the global economy.",
    "The dragon unfurled its wings above the ancient stone castle.",
]

results = classifier(test_samples)
print(f"{'Text':55s}  {'Predicted Genre':30s}  Score")
print("-" * 95)
for text, res in zip(test_samples, results):
    print(f"{text[:53]:55s}  {res['label']:30s}  {res['score']:.4f}")

---
## Summary

| Metric    | Score  |
|-----------|--------|
| Accuracy  | 0.6464 |
| F1 Score  | 0.6452 |
| Eval Loss | 0.9723 |

**Links**
- GitHub: https://github.com/g25ait2004-cyber/Abhishek
- W&B Dashboard: https://wandb.ai/g25ait2032-iit-jodhpur/mlops-assignment2
- Hugging Face: https://huggingface.co/g25ait2004/DistilBERT_Goodreads